In [0]:
%pip install google-api-python-client google-auth


In [0]:
import json
with open("/Volumes/dev_p2p/staging/raw_data/GoogleDriveCredentials/testproject1-501911-eea1aa67b41e.json") as f:
       creds_dict = json.load(f)

In [0]:
from google.oauth2 import service_account
from googleapiclient.discovery import build

# Build credentials from the loaded dict
creds = service_account.Credentials.from_service_account_info(
    creds_dict,
    scopes=["https://www.googleapis.com/auth/drive.readonly"]
)

# Connect to the Drive API
drive_service = build("drive", "v3", credentials=creds)

# Test the connection — list files visible to the service account
results = drive_service.files().list(
    pageSize=20,
    fields="files(id, name, mimeType)"
).execute()

files = results.get("files", [])
if not files:
    print("No files found. Did you share a folder with the service account's email?")
else:
    for f in files:
        print(f"{f['name']}  |  {f['id']}  |  {f['mimeType']}")

In [0]:
import os
import io
from googleapiclient.http import MediaIoBaseDownload

# Map native Google formats to an export mimeType + file extension
GOOGLE_EXPORT_MAP = {
    "application/vnd.google-apps.document": ("application/pdf", ".pdf"),
    "application/vnd.google-apps.spreadsheet": ("text/csv", ".csv"),
    "application/vnd.google-apps.presentation": ("application/pdf", ".pdf"),
}

FOLDER_MIME = "application/vnd.google-apps.folder"

def list_children(folder_id):
    """List all files/folders directly inside a given Drive folder."""
    items = []
    page_token = None
    while True:
        response = drive_service.files().list(
            q=f"'{folder_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, mimeType)",
            pageToken=page_token
        ).execute()
        items.extend(response.get("files", []))
        page_token = response.get("nextPageToken")
        if not page_token:
            break
    return items

def download_file(file_id, mime_type, dest_path):
    """Download or export a single file to dest_path."""
    if mime_type in GOOGLE_EXPORT_MAP:
        export_mime, ext = GOOGLE_EXPORT_MAP[mime_type]
        if not dest_path.endswith(ext):
            dest_path += ext
        request = drive_service.files().export_media(fileId=file_id, mimeType=export_mime)
    elif mime_type in ("application/vnd.google-apps.folder",):
        return  # shouldn't happen, folders handled separately
    else:
        request = drive_service.files().get_media(fileId=file_id)

    buffer = io.BytesIO()
    downloader = MediaIoBaseDownload(buffer, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()

    with open(dest_path, "wb") as f:
        f.write(buffer.getvalue())
    print(f"Downloaded: {dest_path}")

def sync_folder(folder_id, local_path):
    """Recursively replicate a Drive folder's structure and files under local_path."""
    os.makedirs(local_path, exist_ok=True)
    items = list_children(folder_id)

    for item in items:
        item_path = os.path.join(local_path, item["name"])
        if item["mimeType"] == FOLDER_MIME:
            sync_folder(item["id"], item_path)  # recurse into subfolder
        else:
            try:
                download_file(item["id"], item["mimeType"], item_path)
            except Exception as e:
                print(f"Failed to download {item['name']}: {e}")

# --- Run it ---
root_folder_id = "1MpPeFV7PXFuFMdj1sJbfg9FRwFkBIqW5"  # from the Drive URL
volume_base_path = "/Volumes/dev_p2p/staging/raw_data/"

sync_folder(root_folder_id, volume_base_path)